# 02 Run Full (S3PO-GS)

Goal:

1. use the prepared split datasets from `dataset/*/part2_s3po/{sparse,test}`;
2. run formal S3PO full pipeline via single-entry `slam.py`;
3. save outputs under `/home/bzhang512/my_storage_500G/CV_Project/output/part2_s3po`;
4. create/update project-side links under `/home/bzhang512/CV_Project/output/part2_s3po`.

Notes:
- For `405841`, this notebook uses unified waymo input (no scene split).
- The split (`sparse` or `test`) is selected by `SPLIT_NAME`.

In [1]:
# For 405841
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [2]:
from pathlib import Path
import json
import shutil
import subprocess
import time
from datetime import datetime

import yaml

# ============== RUN TARGET ==============
# Options: 'Re10k-1', 'DL3DV-2', '405841'
DATASET_NAME = '405841'

# Options: 'sparse', 'test'
SPLIT_NAME = 'full'
# ========================================

# Runtime toggles
RUN_EXTERNAL_EVAL = True  # Auto-run external eval after SLAM
RUN_FULL = True
TIMEOUT_SEC = 0  # 0 means no timeout

USE_GUI = False
USE_WANDB = False
EVAL_RENDERING = True
COLOR_REFINEMENT = True
GLOBAL_BA = True

# Optional CLI overrides (None = keep config value)
OVERRIDE_BEGIN = None
OVERRIDE_END = None
OVERRIDE_TRACKING_ITR = None
OVERRIDE_WINDOW_SIZE = None
OVERRIDE_PATCH_SIZE = None

CV_ROOT = Path('/home/bzhang512/CV_Project')
S3PO_ROOT = CV_ROOT / 'third_party' / 'S3PO-GS'
PART2_S3PO_ROOT = CV_ROOT / 'part2_s3po'
CONFIG_ROOT = PART2_S3PO_ROOT / 'configs'
CHECK_ROOT = PART2_S3PO_ROOT / 'checks'

# Output layout: <storage>/part2_s3po/<dataset>/<experiment>/...
STORAGE_OUTPUT_ROOT = Path('/home/bzhang512/my_storage_500G/CV_Project/output/part2_s3po')
DATASET_DIR_NAME = DATASET_NAME.lower()
EXPERIMENT_DIR_NAME = f's3po_{DATASET_DIR_NAME}_{SPLIT_NAME}'
RUN_OUTPUT_BASE = STORAGE_OUTPUT_ROOT / DATASET_DIR_NAME / EXPERIMENT_DIR_NAME
PROJECT_OUTPUT_ROOT = CV_ROOT / 'output' / 'part2_s3po'
PROJECT_LINK_ROOT = PROJECT_OUTPUT_ROOT

PYTHON_CANDIDATES = [
    Path('/home/bzhang512/miniconda3/envs/s3po-gs/bin/python'),
    Path('/home/bzhang512/miniconda3/envs/S3PO-GS/bin/python'),
]
S3PO_PYTHON = None
for cand in PYTHON_CANDIDATES:
    if cand.exists():
        S3PO_PYTHON = cand
        break
if S3PO_PYTHON is None:
    S3PO_PYTHON = Path('python3')

for d in [CONFIG_ROOT, CHECK_ROOT, STORAGE_OUTPUT_ROOT, RUN_OUTPUT_BASE, PROJECT_OUTPUT_ROOT, PROJECT_LINK_ROOT]:
    d.mkdir(parents=True, exist_ok=True)

if SPLIT_NAME not in {'sparse', 'test', 'full'}:
    raise ValueError(f'Invalid SPLIT_NAME={SPLIT_NAME}')

print('DATASET_NAME =', DATASET_NAME)
print('SPLIT_NAME =', SPLIT_NAME)
print('S3PO_ROOT =', S3PO_ROOT)
print('S3PO_PYTHON =', S3PO_PYTHON)
print('RUN_OUTPUT_BASE =', RUN_OUTPUT_BASE)
print('PROJECT_LINK_ROOT =', PROJECT_LINK_ROOT)

DATASET_NAME = 405841
SPLIT_NAME = full
S3PO_ROOT = /home/bzhang512/CV_Project/third_party/S3PO-GS
S3PO_PYTHON = /home/bzhang512/miniconda3/envs/s3po-gs/bin/python
RUN_OUTPUT_BASE = /home/bzhang512/my_storage_500G/CV_Project/output/part2_s3po/405841/s3po_405841_full
PROJECT_LINK_ROOT = /home/bzhang512/CV_Project/output/part2_s3po


In [3]:
def read_json(path: Path):
    with path.open('r', encoding='utf-8') as f:
        return json.load(f)


def write_json(path: Path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=True)


def list_pngs(folder: Path):
    return sorted([p for p in folder.iterdir() if p.is_file() and p.suffix.lower() == '.png'])


def patch_slam_cli_args_if_needed(slam_py: Path):
    text = slam_py.read_text(encoding='utf-8')
    has_ns = "add_argument('--ns'" in text or 'add_argument("--ns"' in text
    has_sh = "add_argument('--sh'" in text or 'add_argument("--sh"' in text

    if has_ns and has_sh:
        return False

    anchor = "    parser.add_argument(\"--patch_size\", type=int, default=None, help=\"patch size\")\n"
    if anchor not in text:
        raise RuntimeError('Cannot find parser patch_size anchor in slam.py; patch aborted.')

    insert_lines = []
    if not has_ns:
        insert_lines.append("    parser.add_argument('--ns', type=int, default=None, help='mapping_itr_nosingle override')\n")
    if not has_sh:
        insert_lines.append("    parser.add_argument('--sh', type=int, default=None, help='spherical harmonics degree override')\n")

    patched = text.replace(anchor, anchor + ''.join(insert_lines))
    slam_py.write_text(patched, encoding='utf-8')
    return True

## 1. Resolve split input and base config

In [4]:
CONFIG_MAP = {
    ('Re10k-1', 'sparse'): CONFIG_ROOT / 's3po_re10k1_sparse.yaml',
    ('Re10k-1', 'test'): CONFIG_ROOT / 's3po_re10k1_test.yaml',
    ('Re10k-1', 'full'): CONFIG_ROOT / 's3po_re10k1_full.yaml',
    ('DL3DV-2', 'sparse'): CONFIG_ROOT / 's3po_dl3dv2_sparse.yaml',
    ('DL3DV-2', 'test'): CONFIG_ROOT / 's3po_dl3dv2_test.yaml',
    ('DL3DV-2', 'full'): CONFIG_ROOT / 's3po_dl3dv2_full.yaml',
    ('405841', 'sparse'): CONFIG_ROOT / 's3po_405841_sparse_waymo.yaml',
    ('405841', 'test'): CONFIG_ROOT / 's3po_405841_test_waymo.yaml',
    ('405841', 'full'): CONFIG_ROOT / 's3po_405841_full_waymo.yaml',
}

if (DATASET_NAME, SPLIT_NAME) not in CONFIG_MAP:
    raise ValueError(f'Unsupported selection: {(DATASET_NAME, SPLIT_NAME)}')

base_cfg_path = CONFIG_MAP[(DATASET_NAME, SPLIT_NAME)]
assert base_cfg_path.exists(), f'Missing base split config: {base_cfg_path}'

base_cfg = yaml.safe_load(base_cfg_path.read_text(encoding='utf-8'))
dataset_path = Path(base_cfg['Dataset']['dataset_path'])
assert dataset_path.exists(), f'Missing dataset_path: {dataset_path}'

print('base_cfg_path =', base_cfg_path)
print('dataset_path =', dataset_path)
print('loader type =', base_cfg['Dataset']['type'])
print('begin/end =', base_cfg['Dataset'].get('begin', None), base_cfg['Dataset'].get('end', None))

base_cfg_path = /home/bzhang512/CV_Project/part2_s3po/configs/s3po_405841_full_waymo.yaml
dataset_path = /home/bzhang512/CV_Project/dataset/405841/part2_s3po/full
loader type = waymo
begin/end = None None


## 2. Validate input split structure

In [5]:
loader_type = base_cfg['Dataset']['type']

if loader_type in {'dl3dv', 'KITTI'}:
    assert (dataset_path / 'rgb').exists(), 'Missing rgb dir'
    assert (dataset_path / 'cameras.json').exists(), 'Missing cameras.json'
    rgb_pngs = list_pngs(dataset_path / 'rgb')
    assert len(rgb_pngs) > 0, f'No png files in {dataset_path / "rgb"}'

    cameras = read_json(dataset_path / 'cameras.json')
    assert len(cameras) == len(rgb_pngs), 'camera/rgb count mismatch'

    names_rgb = {p.name for p in rgb_pngs}
    names_cam = {c['image_name'] for c in cameras}
    assert names_rgb == names_cam, 'camera image_name mismatch with rgb files'

    print('rgb count =', len(rgb_pngs))
    print('first rgb =', rgb_pngs[0].name)

elif loader_type == 'waymo':
    for sub in ['rgb', 'depth', 'mono_depth', 'gt']:
        assert (dataset_path / sub).exists(), f'Missing {sub} dir'

    rgb_pngs = list_pngs(dataset_path / 'rgb')
    dep_pngs = list_pngs(dataset_path / 'depth')
    mono_pngs = list_pngs(dataset_path / 'mono_depth')
    gt_txts = sorted([p for p in (dataset_path / 'gt').iterdir() if p.is_file() and p.suffix.lower() == '.txt'])

    assert len(rgb_pngs) > 0, 'Empty waymo rgb split'
    assert len(rgb_pngs) == len(dep_pngs) == len(mono_pngs) == len(gt_txts), 'waymo split count mismatch'

    rgb_stems = {p.stem for p in rgb_pngs}
    dep_stems = {p.stem for p in dep_pngs}
    mono_stems = {p.stem for p in mono_pngs}
    gt_stems = {p.stem for p in gt_txts}
    assert rgb_stems == dep_stems == mono_stems == gt_stems, 'waymo stem mismatch across folders'

    print('waymo split count =', len(rgb_pngs))
    print('first rgb stem =', rgb_pngs[0].stem)

else:
    raise ValueError(f'Unsupported loader_type={loader_type}')

waymo split count = 199
first rgb stem = 000000


## 3. Build formal run config

In [6]:
run_cfg = json.loads(json.dumps(base_cfg))

dataset_tag = DATASET_NAME.lower().replace('-', '')
run_tag = f's3po_{dataset_tag}_{SPLIT_NAME}_full'
run_cfg_path = CONFIG_ROOT / f'{run_tag}.yaml'

run_cfg.setdefault('Results', {})
run_cfg['Results']['save_results'] = True
run_cfg['Results']['save_dir'] = str(RUN_OUTPUT_BASE)
run_cfg['Results']['use_gui'] = bool(USE_GUI)
run_cfg['Results']['use_wandb'] = bool(USE_WANDB)
run_cfg['Results']['eval_rendering'] = bool(EVAL_RENDERING)
run_cfg['Results']['color_refinement'] = bool(COLOR_REFINEMENT)
run_cfg['Results']['global_BA'] = bool(GLOBAL_BA)

# important for ATE evaluation
run_cfg['Training']['kf_interval'] = 10
run_cfg['Training']['tracking_itr_num'] = 300
run_cfg['Training']['window_size'] = 10
run_cfg['Training']['mapping_itr_num'] = 300
run_cfg['Training']['pose_window'] = 5
run_cfg['Training']['alpha'] = 0.975
run_cfg['Training']['global_BA_itr_num'] = 600

# Auto-set force_keyframe_indices based on SPLIT_NAME
if SPLIT_NAME == 'sparse':
    # Sparse: all frames become keyframes
    run_cfg['Training']['force_keyframe_indices'] = list(range(len(rgb_pngs)))
    print(f'[Sparse mode] force_keyframe_indices = range({len(rgb_pngs)})')
else:
    # Full/Test: use sparse selected_indices from manifest
    dataset_root = Path(run_cfg['Dataset']['dataset_path']).parent
    sparse_manifest_path = dataset_root / 'sparse' / 'split_manifest.json'
    if sparse_manifest_path.exists():
        sparse_manifest = read_json(sparse_manifest_path)
        run_cfg['Training']['force_keyframe_indices'] = sparse_manifest.get('selected_indices', [])
        print(f'[{SPLIT_NAME} mode] force_keyframe_indices from sparse manifest: {len(run_cfg["Training"]["force_keyframe_indices"])} frames')
    else:
        print(f'[WARN] No sparse manifest at {sparse_manifest_path}')

if OVERRIDE_BEGIN is not None:
    run_cfg['Dataset']['begin'] = int(OVERRIDE_BEGIN)
if OVERRIDE_END is not None:
    run_cfg['Dataset']['end'] = int(OVERRIDE_END)
if OVERRIDE_TRACKING_ITR is not None:
    run_cfg['Training']['tracking_itr_num'] = int(OVERRIDE_TRACKING_ITR)
if OVERRIDE_WINDOW_SIZE is not None:
    run_cfg['Training']['window_size'] = int(OVERRIDE_WINDOW_SIZE)
if OVERRIDE_PATCH_SIZE is not None:
    run_cfg['depth']['patch_size'] = int(OVERRIDE_PATCH_SIZE)

run_cfg_path.write_text(yaml.safe_dump(run_cfg, sort_keys=False), encoding='utf-8')

dataset_path_parts = Path(run_cfg['Dataset']['dataset_path']).parts
save_group_name = f'{dataset_path_parts[-3]}_{dataset_path_parts[-2]}'
output_group_dir = RUN_OUTPUT_BASE / save_group_name

plan_json = CHECK_ROOT / f'{run_tag}_plan.json'
write_json(plan_json, {
    'dataset_name': DATASET_NAME,
    'split_name': SPLIT_NAME,
    'loader_type': loader_type,
    'base_cfg_path': str(base_cfg_path),
    'run_cfg_path': str(run_cfg_path),
    'run_tag': run_tag,
    'dataset_path': run_cfg['Dataset']['dataset_path'],
    'output_base': str(RUN_OUTPUT_BASE),
    'output_group_dir': str(output_group_dir),
})

print('run_tag =', run_tag)
print('run_cfg_path =', run_cfg_path)
print('output_group_dir =', output_group_dir)
print('plan_json =', plan_json)
print(run_cfg_path.read_text(encoding='utf-8'))

[full mode] force_keyframe_indices from sparse manifest: 20 frames
run_tag = s3po_405841_full_full
run_cfg_path = /home/bzhang512/CV_Project/part2_s3po/configs/s3po_405841_full_full.yaml
output_group_dir = /home/bzhang512/my_storage_500G/CV_Project/output/part2_s3po/405841/s3po_405841_full/405841_part2_s3po
plan_json = /home/bzhang512/CV_Project/part2_s3po/checks/s3po_405841_full_full_plan.json
Results:
  save_results: true
  save_dir: /home/bzhang512/my_storage_500G/CV_Project/output/part2_s3po/405841/s3po_405841_full
  save_trj: true
  save_trj_kf_intv: 10
  use_gui: false
  eval_rendering: true
  use_wandb: false
  color_refinement: true
  global_BA: true
Dataset:
  type: waymo
  sensor_type: monocular
  depth_loss: true
  pcd_downsample: 64
  pcd_downsample_init: 32
  adaptive_pointsize: true
  point_size: 0.01
  dataset_path: /home/bzhang512/CV_Project/dataset/405841/part2_s3po/full
  Calibration:
    fx: 551.1193505112797
    fy: 826.6790257669196
    cx: 253.48034064401926
    c

## 4. Optional cleanup

In [7]:
# Optional cleanup of latest output group before run
RESET_OUTPUT_GROUP = False
if RESET_OUTPUT_GROUP and output_group_dir.exists():
    shutil.rmtree(output_group_dir)
    print('Removed output group:', output_group_dir)
else:
    print('RESET_OUTPUT_GROUP =', RESET_OUTPUT_GROUP)
    print('output_group_dir =', output_group_dir)

RESET_OUTPUT_GROUP = False
output_group_dir = /home/bzhang512/my_storage_500G/CV_Project/output/part2_s3po/405841/s3po_405841_full/405841_part2_s3po


## 5. Run full pipeline

In [8]:
slam_py = S3PO_ROOT / 'slam.py'
patched = patch_slam_cli_args_if_needed(slam_py)
print('Patched slam.py (--ns/--sh) =', patched)

cmd = [str(S3PO_PYTHON), 'slam.py', '--config', str(run_cfg_path)]
if OVERRIDE_BEGIN is not None:
    cmd += ['--begin', str(int(OVERRIDE_BEGIN))]
if OVERRIDE_END is not None:
    cmd += ['--end', str(int(OVERRIDE_END))]
if OVERRIDE_TRACKING_ITR is not None:
    cmd += ['--iter', str(int(OVERRIDE_TRACKING_ITR))]
if OVERRIDE_WINDOW_SIZE is not None:
    cmd += ['--windowsize', str(int(OVERRIDE_WINDOW_SIZE))]
if OVERRIDE_PATCH_SIZE is not None:
    cmd += ['--patch_size', str(int(OVERRIDE_PATCH_SIZE))]

print('cmd =', ' '.join(cmd))

proc = None
elapsed = None
if RUN_FULL:
    start_t = time.time()
    try:
        # Stream logs in real time to avoid waiting until process exits.
        with subprocess.Popen(
            cmd,
            cwd=str(S3PO_ROOT),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        ) as p:
            proc = p
            for line in p.stdout:
                print(line, end='')
            if TIMEOUT_SEC and TIMEOUT_SEC > 0:
                p.wait(timeout=TIMEOUT_SEC)
            else:
                p.wait()

        elapsed = time.time() - start_t
        print('returncode =', proc.returncode)
        print('elapsed_sec =', elapsed)

        if proc.returncode != 0:
            print('[WARN] Full run finished with non-zero return code.')

    except subprocess.TimeoutExpired:
        elapsed = time.time() - start_t
        if proc is not None:
            proc.kill()
            proc.wait()
        print(f'[ERROR] Timeout after {TIMEOUT_SEC} sec.')
        print('elapsed_sec =', elapsed)

    except FileNotFoundError as e:
        elapsed = time.time() - start_t
        print('[ERROR] Executable/script not found:', e)
        print('Check S3PO_PYTHON and slam.py path.')
        print('elapsed_sec =', elapsed)

    except KeyboardInterrupt:
        elapsed = time.time() - start_t
        if proc is not None:
            proc.kill()
            proc.wait()
        print('[INTERRUPTED] Run stopped by user.')
        print('elapsed_sec =', elapsed)

    except Exception as e:
        elapsed = time.time() - start_t
        if proc is not None and proc.poll() is None:
            proc.kill()
            proc.wait()
        print('[ERROR] Unexpected exception during run:', repr(e))
        print('elapsed_sec =', elapsed)
else:
    print('Set RUN_FULL=True to execute full run.')

Patched slam.py (--ns/--sh) = False
cmd = /home/bzhang512/miniconda3/envs/s3po-gs/bin/python slam.py --config /home/bzhang512/CV_Project/part2_s3po/configs/s3po_405841_full_full.yaml
Current backend: agg
Warning, cannot find cuda-compiled version of RoPE2D, using a slow pytorch version instead
S3PO-GS: saving results in 
/home/bzhang512/my_storage_500G/CV_Project/output/part2_s3po/405841/s3po_405841_
full/405841_part2_s3po/2026-04-04-00-43-20
... loading model from naver/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric
instantiating : AsymmetricMASt3R(enc_depth=24, dec_depth=12, enc_embed_dim=1024, dec_embed_dim=768, enc_num_heads=16, dec_num_heads=12, pos_embed='RoPE100',img_size=(512, 512), head_type='catmlp+dpt', output_mode='pts3d+desc24', depth_mode=('exp', -inf, inf), conf_mode=('exp', 1, inf), patch_embed_cls='PatchEmbedDust3R', two_confs=True, desc_conf_mode=('exp', 0, inf), landscape_only=False)
<All keys matched successfully>
Initial depth map stats for frame 0 : Max: 13.3029

## 6. Post-run checks and project-side link

In [9]:
latest_run_dir = None
if output_group_dir.exists():
    run_dirs = sorted([p for p in output_group_dir.iterdir() if p.is_dir()], key=lambda p: p.stat().st_mtime)
    if run_dirs:
        latest_run_dir = run_dirs[-1]

print('output_group_dir exists =', output_group_dir.exists())
print('latest_run_dir =', latest_run_dir)

if latest_run_dir is not None:
    print('top-level entries:')
    for p in sorted(latest_run_dir.iterdir())[:30]:
        print(' -', p.name)

    # link_path = PROJECT_LINK_ROOT / f'{run_tag}_latest'
    # if link_path.is_symlink():
    #     link_path.unlink()
    #     link_path.symlink_to(latest_run_dir)
    #     print('Updated link:', link_path, '->', latest_run_dir)
    # elif link_path.exists():
    #     print('[WARN] Link path exists and is not symlink, skip:', link_path)
    # else:
    #     link_path.symlink_to(latest_run_dir)
    #     print('Created link:', link_path, '->', latest_run_dir)

    plot_stats = list(latest_run_dir.glob('plot/stats_*.json'))
    traj_json = list(latest_run_dir.glob('plot/trj_*.json'))
    psnr_json = list(latest_run_dir.glob('psnr/**/final_result.json'))
    ply_final = latest_run_dir / 'point_cloud' / 'final' / 'point_cloud.ply'

    print('plot stats files =', [p.name for p in plot_stats])
    print('trajectory files =', [p.name for p in traj_json])
    print('psnr files =', [str(p.relative_to(latest_run_dir)) for p in psnr_json])
    print('final ply exists =', ply_final.exists())

    result_json = CHECK_ROOT / f'{run_tag}_result.json'
    write_json(result_json, {
        'run_tag': run_tag,
        'dataset_name': DATASET_NAME,
        'split_name': SPLIT_NAME,
        'loader_type': loader_type,
        'run_cfg_path': str(run_cfg_path),
        'output_group_dir': str(output_group_dir),
        'latest_run_dir': str(latest_run_dir),
        # 'project_link': str(link_path),
        'plot_stats_files': [p.name for p in plot_stats],
        'trajectory_files': [p.name for p in traj_json],
        'psnr_files': [str(p.relative_to(latest_run_dir)) for p in psnr_json],
        'final_ply_exists': bool(ply_final.exists()),
        'returncode': None if proc is None else int(proc.returncode),
    })
    print('result_json =', result_json)
else:
    print('No run directory found under:', output_group_dir)

output_group_dir exists = True
latest_run_dir = /home/bzhang512/my_storage_500G/CV_Project/output/part2_s3po/405841/s3po_405841_full/405841_part2_s3po/2026-04-04-00-43-20
top-level entries:
 - config.yml
 - plot
 - point_cloud
 - psnr
 - render_depth
 - render_depth_npy
 - render_rgb
 - viz
plot stats files = ['stats_0094.json', 'stats_0199.json', 'stats_final.json']
trajectory files = ['trj_final.json', 'trj_0094.json', 'trj_0199.json']
psnr files = ['psnr/before_opt/final_result.json', 'psnr/after_opt/final_result.json']
final ply exists = True
result_json = /home/bzhang512/CV_Project/part2_s3po/checks/s3po_405841_full_full_result.json


In [10]:
# === Run External Eval (auto) ===
if RUN_EXTERNAL_EVAL and latest_run_dir is not None:
    # 找 ply
    ply_path = latest_run_dir / 'point_cloud' / 'final' / 'point_cloud.ply'
    if not ply_path.exists():
        print(f'[WARN] Missing ply: {ply_path}')
    else:
        # 直接用固定路径的 test config
        test_config_map = {
            'Re10k-1': CONFIG_ROOT / 's3po_re10k1_test.yaml',
            'DL3DV-2': CONFIG_ROOT / 's3po_dl3dv2_test.yaml',
            '405841': CONFIG_ROOT / 's3po_405841_test.yaml',
        }
        test_config = test_config_map.get(DATASET_NAME)
        if test_config is None or not test_config.exists():
            print(f'[WARN] Missing test config for {DATASET_NAME}')
        else:
            # 提取 SLAM 时间戳
            slam_timestamp = latest_run_dir.name  # e.g., '2026-04-03-12-48-45'
            
            # 构建保存目录
            # e.g., 00_external_eval/re10k1/infer_re10k1_full/2026-04-03-12-48-45/
            pose_mode = 'infer'
            eval_run_name = f'{pose_mode}_{dataset_tag}_{SPLIT_NAME}'
            external_output = STORAGE_OUTPUT_ROOT / '00_external_eval' / dataset_tag / eval_run_name / slam_timestamp
            external_output.mkdir(parents=True, exist_ok=True)
            
            print(f'External eval config: {test_config}')
            print(f'External eval ply: {ply_path}')
            print(f'External eval output: {external_output}')
            
            # 调用 eval_external.py
            cmd = [
                str(S3PO_PYTHON),
                str(S3PO_ROOT / 'eval_external.py'),
                '--config', str(test_config),
                '--ply_path', str(ply_path),
                '--save_dir', str(external_output),
                '--pose_mode', pose_mode,
                '--origin_mode', 'test_to_sparse_first',
                '--infer_init_mode', 'gt_first',
                '--save_render_depth',
                '--save_render_depth_npy',
            ]
            
            env = dict(os.environ)
            env['MPLBACKEND'] = 'Agg'
            
            proc = subprocess.run(
                cmd,
                cwd=str(S3PO_ROOT),
                env=env,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
            )
            print(proc.stdout)
            print(f'External eval returncode = {proc.returncode}')
            
            # 检查结果
            eval_json = external_output / 'eval_external.json'
            if eval_json.exists():
                eval_result = json.loads(eval_json.read_text())
                print(f'Infer avg_psnr = {eval_result.get("avg_psnr", None)}')
                print(f'Infer avg_ssim = {eval_result.get("avg_ssim", None)}')
                print(f'Infer avg_lpips = {eval_result.get("avg_lpips", None)}')
                pose_info = eval_result.get('pose', {})
                print(f'Infer ate_rmse = {pose_info.get("ate_rmse", None)}')
else:
    print('RUN_EXTERNAL_EVAL=False or no latest_run_dir')


External eval config: /home/bzhang512/CV_Project/part2_s3po/configs/s3po_405841_test.yaml
External eval ply: /home/bzhang512/my_storage_500G/CV_Project/output/part2_s3po/405841/s3po_405841_full/405841_part2_s3po/2026-04-04-00-43-20/point_cloud/final/point_cloud.ply
External eval output: /home/bzhang512/my_storage_500G/CV_Project/output/part2_s3po/00_external_eval/405841/infer_405841_full/2026-04-04-00-43-20
Warning, cannot find cuda-compiled version of RoPE2D, using a slow pytorch version instead
Current backend: agg
Eval: Could not compute test-to-sparse origin offset; fallback to no offset.
... loading model from naver/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric
instantiating : AsymmetricMASt3R(enc_depth=24, dec_depth=12, enc_embed_dim=1024, dec_embed_dim=768, enc_num_heads=16, dec_num_heads=12, pos_embed='RoPE100',img_size=(512, 512), head_type='catmlp+dpt', output_mode='pts3d+desc24', depth_mode=('exp', -inf, inf), conf_mode=('exp', 1, inf), patch_embed_cls='PatchEmbedDust3R',